# Day 2 · Exercise 3: Which DCs should Nike India open?

**What you'll be able to say by the end**
1. "The best network opens **Delhi, Mumbai, Bengaluru and Kolkata** for ₹33.8 lakh a week. Nagpur, the cheapest and most central site, is *not* in it."
2. "A **MIP** is an LP with some yes/no decisions. The solver gets lower bounds from **LP relaxations** and searches with **branch and bound**."
3. "A **1% MIP gap** means the plan we have is at most 1% more expensive than the best plan possible."
4. "How you write the model matters. One extra rule cut the search from 31 LPs to 1."

**New ideas today:** binary (yes/no) variables, fixed costs, the linking or *big-M* constraint, LP relaxation, branch and bound, MIP gap, and formulation strength.

_Setup check: make sure this notebook runs on the repo's `.venv`. Nothing to learn here._

In [1]:
import sys
from pathlib import Path

VENV = (Path.cwd().parent / ".venv").resolve()
assert Path(sys.prefix).resolve() == VENV, f"Wrong kernel: {sys.prefix}. Select 'supply-chain-or (.venv)'."
print("Kernel:", sys.executable)

Kernel: /Users/abhijeet/Personal/supply_chain_or/.venv/bin/python


## 1. The story

Nike India wants a distribution network. There are **5 candidate DC sites**. Each one:
- costs a **fixed amount every week** if it's open (rent, staff, security), whether it ships 1 pair or 30,000;
- can handle only so many pairs a week (its **capacity**).

There are **20 cities** to supply. Shipping a pair costs ₹5 handling plus ₹0.05 per road-km.

**The question:** which sites should we open, and which DC should serve each city, to keep the weekly cost (fixed + shipping) as low as possible?

(1 lakh = ₹1,00,000.)

In [2]:
import itertools
import math
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import pulp
from ortools.linear_solver import pywraplp
from plotly.subplots import make_subplots

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

cities = {  # name: (latitude, longitude, pairs needed per week)
    "Delhi": (28.61, 77.21, 9000), "Mumbai": (19.08, 72.88, 9000), "Bengaluru": (12.97, 77.59, 7000),
    "Kolkata": (22.57, 88.36, 4500), "Chennai": (13.08, 80.27, 5000), "Hyderabad": (17.39, 78.49, 5000),
    "Ahmedabad": (23.02, 72.57, 3500), "Pune": (18.52, 73.86, 4000), "Jaipur": (26.91, 75.79, 2000),
    "Lucknow": (26.85, 80.95, 2000), "Chandigarh": (30.73, 76.78, 1500), "Kochi": (9.93, 76.27, 1500),
    "Bhubaneswar": (20.30, 85.82, 1200), "Guwahati": (26.14, 91.74, 1000), "Indore": (22.72, 75.86, 1500),
    "Nagpur": (21.15, 79.09, 1200), "Patna": (25.59, 85.14, 1300), "Surat": (21.17, 72.83, 2000),
    "Coimbatore": (11.02, 76.96, 1200), "Visakhapatnam": (17.69, 83.22, 1100),
}
CITIES = list(cities)
demand = {c: cities[c][2] for c in CITIES}

SITES = ["Delhi", "Mumbai", "Bengaluru", "Kolkata", "Nagpur"]
fixed_cost = {"Delhi": 600_000, "Mumbai": 700_000, "Bengaluru": 550_000,
              "Kolkata": 400_000, "Nagpur": 300_000}                     # Rs per week if open
capacity = {"Delhi": 30_000, "Mumbai": 30_000, "Bengaluru": 25_000,
            "Kolkata": 20_000, "Nagpur": 35_000}                         # pairs per week


def road_km(a, b):
    """Straight-line (great-circle) distance x 1.25, a common rule of thumb for road distance."""
    lat1, lon1, _ = cities[a]
    lat2, lon2, _ = cities[b]
    p1, p2 = math.radians(lat1), math.radians(lat2)
    h = (math.sin((p2 - p1) / 2) ** 2
         + math.cos(p1) * math.cos(p2) * math.sin(math.radians(lon2 - lon1) / 2) ** 2)
    return 1.25 * 2 * 6371 * math.asin(math.sqrt(h))


ship_cost = pd.DataFrame({c: {s: 5 + 0.05 * road_km(s, c) for s in SITES} for c in CITIES})  # Rs per pair


def lakh(rupees):
    return f"Rs {rupees / 1e5:,.1f} lakh"


pd.DataFrame({"fixed cost per week": {s: lakh(v) for s, v in fixed_cost.items()},
              "capacity (pairs/week)": capacity})

,fixed cost per week,capacity (pairs/week)
Delhi,Rs 6.0 lakh,30000
Mumbai,Rs 7.0 lakh,30000
Bengaluru,Rs 5.5 lakh,25000
Kolkata,Rs 4.0 lakh,20000
Nagpur,Rs 3.0 lakh,35000


In [3]:
def india_map(title, lines=()):
    """Cities sized by demand; candidate DC sites as diamonds; optional (site, city) lines."""
    fig = go.Figure()
    for s, c in lines:
        fig.add_trace(go.Scattergeo(lat=[cities[s][0], cities[c][0]], lon=[cities[s][1], cities[c][1]],
                                    mode="lines", line=dict(width=1, color="gray"), hoverinfo="skip"))
    fig.add_trace(go.Scattergeo(
        lat=[cities[c][0] for c in CITIES], lon=[cities[c][1] for c in CITIES], text=CITIES,
        marker=dict(size=[4 + demand[c] / 400 for c in CITIES], color="#4C78A8", opacity=0.7),
        customdata=[demand[c] for c in CITIES], name="city",
        hovertemplate="%{text}<br>%{customdata:,} pairs/week<extra></extra>"))
    fig.add_trace(go.Scattergeo(
        lat=[cities[s][0] for s in SITES], lon=[cities[s][1] for s in SITES], text=SITES,
        mode="markers+text", textposition="top right",
        marker=dict(size=14, symbol="diamond", color="#E45756", line=dict(width=1, color="white")),
        name="candidate DC", hovertemplate="%{text} DC site<extra></extra>"))
    fig.update_geos(fitbounds="locations", showcountries=True, countrycolor="lightgray",
                    showland=True, landcolor="#f7f7f7")
    fig.update_layout(title=title, height=560, width=620, showlegend=False, margin=dict(l=0, r=0, t=40, b=0))
    fig.show()

india_map("Cities (blue, sized by demand) and candidate DC sites (red)")

**Things to notice:**
- Demand is concentrated in the big metros: Delhi, Mumbai and Bengaluru need 25,000 of the 64,500 pairs.
- **Nagpur** is right in the middle of the country and is the cheapest site (₹3 lakh a week), with the biggest capacity. It looks like the obvious pick.
- **Mumbai** is the most expensive site (₹7 lakh a week).

Here's what it costs to ship one pair from each site to each city (₹):

In [4]:
ship_cost.round(0).astype(int)

,Delhi,Mumbai,Bengaluru,Kolkata,Chennai,Hyderabad,Ahmedabad,Pune,Jaipur,Lucknow,Chandigarh,Kochi,Bhubaneswar,Guwahati,Indore,Nagpur,Patna,Surat,Coimbatore,Visakhapatnam
Delhi,5,77,114,86,115,83,53,78,20,31,20,135,84,96,47,58,58,64,127,90
Mumbai,77,5,58,108,70,44,32,13,63,80,90,73,90,135,37,48,96,20,67,74
Bengaluru,114,58,5,103,23,36,82,51,103,104,129,28,80,135,74,63,106,70,19,55
Kolkata,86,108,103,5,90,79,106,103,90,60,96,124,28,38,85,66,34,106,115,53
Nagpur,58,48,63,66,62,31,49,44,50,46,73,85,49,93,29,5,54,46,77,41


## 2. Why this isn't an LP any more

In Day 1, every decision was a quantity that could be any number: 156.25 pairs was fine. Here, the main decision is **yes or no**: a DC is open or it isn't. You can't open 0.4 of a warehouse and pay 40% of the rent.

So we need a new kind of variable: a **binary variable**, which can only be **0 (closed)** or **1 (open)**. An LP that also has some whole-number variables is called a **mixed-integer program (MIP)**.

This one choice changes everything about how hard the problem is to solve, as we'll see.

## 3. Solve it by hand: try every combination

With 5 sites there are only 2⁵ − 1 = **31** ways to open at least one DC. For each combination, the rest of the problem is just **Day 1's transportation problem**: ship from the open DCs to the cities as cheaply as possible. So we can try all 31, add each one's fixed cost, and pick the cheapest.

In [5]:
def shipping_only(open_sites):
    """Day 1's transportation problem: cheapest shipping from a fixed set of open DCs."""
    if sum(capacity[s] for s in open_sites) < sum(demand.values()):
        return None  # not enough capacity to serve everyone
    model = pulp.LpProblem("shipping", pulp.LpMinimize)
    ship = {(s, c): model.add_variable(f"ship_{s}_{c}", lowBound=0) for s in open_sites for c in CITIES}
    model += pulp.lpSum(ship_cost.loc[s, c] * ship[s, c] for s, c in ship)
    for c in CITIES:
        model += pulp.lpSum(ship[s, c] for s in open_sites) >= demand[c]
    for s in open_sites:
        model += pulp.lpSum(ship[s, c] for c in CITIES) <= capacity[s]
    model.solve(pulp.HiGHS(msg=False))
    return pulp.value(model.objective)


rows = []
for k in range(1, len(SITES) + 1):
    for combo in itertools.combinations(SITES, k):
        shipping = shipping_only(combo)
        fixed = sum(fixed_cost[s] for s in combo)
        rows.append({"open DCs": " + ".join(combo), "fixed": fixed, "shipping": shipping,
                     "total": None if shipping is None else fixed + shipping})
combos = pd.DataFrame(rows)
feasible = combos.dropna().sort_values("total").reset_index(drop=True)

print(f"{len(combos)} combinations; {len(feasible)} have enough capacity for all {sum(demand.values()):,} pairs")
feasible.head(6).assign(**{k: feasible.head(6)[k].map(lakh) for k in ["fixed", "shipping", "total"]})

31 combinations; 18 have enough capacity for all 64,500 pairs


,open DCs,fixed,shipping,total
0,Delhi + Mumbai + Bengaluru + Kolkata,Rs 22.5 lakh,Rs 11.3 lakh,Rs 33.8 lakh
1,Delhi + Bengaluru + Nagpur,Rs 14.5 lakh,Rs 20.3 lakh,Rs 34.8 lakh
2,Delhi + Bengaluru + Kolkata + Nagpur,Rs 18.5 lakh,Rs 16.5 lakh,Rs 35.0 lakh
3,Delhi + Mumbai + Bengaluru,Rs 18.5 lakh,Rs 16.5 lakh,Rs 35.0 lakh
4,Delhi + Mumbai + Bengaluru + Nagpur,Rs 21.5 lakh,Rs 14.1 lakh,Rs 35.6 lakh
5,Delhi + Mumbai + Bengaluru + Kolkata + Nagpur,Rs 25.5 lakh,Rs 10.3 lakh,Rs 35.8 lakh


In [6]:
fig = go.Figure()
fig.add_trace(go.Bar(x=feasible["open DCs"], y=feasible["fixed"] / 1e5, name="fixed cost",
                     hovertemplate="%{x}<br>fixed Rs %{y:.1f} lakh<extra></extra>"))
fig.add_trace(go.Bar(x=feasible["open DCs"], y=feasible["shipping"] / 1e5, name="shipping cost",
                     hovertemplate="%{x}<br>shipping Rs %{y:.1f} lakh<extra></extra>"))
fig.update_layout(barmode="stack", title="Every workable combination, cheapest first",
                  yaxis_title="Rs lakh per week", height=480, width=1000, template="plotly_white",
                  xaxis_tickangle=-40, legend=dict(orientation="h", y=1.08))
fig.show()

**The winner opens 4 DCs: Delhi, Mumbai, Bengaluru and Kolkata (₹33.8 lakh a week).** Nagpur isn't in it.

Compare it with the runner-up, **Delhi + Bengaluru + Nagpur** (₹34.8 lakh):

| | Fixed (rent) | Shipping | Total |
|---|---|---|---|
| Delhi + Mumbai + Bengaluru + Kolkata | ₹22.5 lakh | ₹11.3 lakh | **₹33.8 lakh** |
| Delhi + Bengaluru + Nagpur | ₹14.5 lakh | ₹20.3 lakh | ₹34.8 lakh |

The Nagpur plan saves ₹8 lakh in rent but spends ₹9 lakh more on shipping. Being in the middle of the country means being **medium-far from everyone**. A DC *inside* a big metro ships its own city's pairs for ₹5, and Mumbai alone needs 9,000 pairs a week. Nagpur city needs only 1,200.

That's the core tradeoff in every network design question: **more DCs = more fixed cost, less shipping**. The chart shows it: going from left to right, the blue (fixed) and red (shipping) parts trade off against each other.

**The catch:** trying every combination works for 5 sites (31 tries). With 20 sites it's over a million tries, and with 50 sites about 10¹⁵. Real networks need a smarter method.

## 4. Write it as a MIP

| Piece | Plain words | Here |
|---|---|---|
| **Binary variables** | Yes/no choices | `open[s]` = 1 if site *s* is open, 0 if not |
| **Continuous variables** | Quantities, like Day 1 | `ship[s, c]` = pairs from site *s* to city *c* |
| **Objective** | Minimize | Σ fixed cost × `open` + Σ shipping cost × `ship` |
| **Demand rule** | Every city gets what it needs | Σ over sites of `ship[s, c]` ≥ demand of *c* |
| **Linking rule** | A closed DC ships nothing; an open one ships up to capacity | Σ over cities of `ship[s, c]` ≤ capacity × `open[s]` |

**The linking rule is the key trick.** Read it with each value of `open`:
- `open = 0`: right-hand side is capacity × 0 = 0, so the site can't ship anything.
- `open = 1`: right-hand side is capacity × 1, so it's the normal capacity limit.

One rule connects the yes/no decision to the quantities. This pattern, "quantity ≤ M × binary", is called a **big-M constraint**, and here **M** is the capacity. You'll see it in almost every MIP (it's an interview favorite: exercise 4 shows what goes wrong when M is too big).

Because `open` is multiplied by the fixed cost in the objective, you pay the full rent if `open = 1` and nothing if it's 0. That's how a MIP handles a **fixed cost**.

In [7]:
def solve_mip(relax=False, tight=False, fixed=None):
    """Facility location. relax: 'open' may be fractional. tight: add ship <= demand x open.
    fixed: {site: 0 or 1} to force some sites closed/open (used by branch and bound)."""
    model = pulp.LpProblem("facility_location", pulp.LpMinimize)

    # Decision variables
    is_open = {s: model.add_variable(f"open_{s}", lowBound=0, upBound=1,
                                     cat="Continuous" if relax else "Binary") for s in SITES}
    ship = {(s, c): model.add_variable(f"ship_{s}_{c}", lowBound=0) for s in SITES for c in CITIES}

    # Objective: fixed cost of open DCs + shipping cost
    model += (pulp.lpSum(fixed_cost[s] * is_open[s] for s in SITES)
              + pulp.lpSum(ship_cost.loc[s, c] * ship[s, c] for s in SITES for c in CITIES))

    # Every city gets what it needs
    for c in CITIES:
        model += pulp.lpSum(ship[s, c] for s in SITES) >= demand[c], f"demand_{c}"

    # Linking constraint: a closed DC ships nothing; an open one ships up to its capacity
    for s in SITES:
        model += pulp.lpSum(ship[s, c] for c in CITIES) <= capacity[s] * is_open[s], f"capacity_{s}"

    # Tighter version: a DC can't send a city more than that city needs, and nothing if closed
    if tight:
        for s in SITES:
            for c in CITIES:
                model += ship[s, c] <= demand[c] * is_open[s]

    for s, v in (fixed or {}).items():
        model += is_open[s] == v

    stats = model.solve(pulp.HiGHS(msg=False))
    return stats, model, is_open, ship


stats, model, is_open, ship = solve_mip()
mip_cost = pulp.value(model.objective)
mip_open = [s for s in SITES if is_open[s].varValue > 0.5]
print(f"MIP says open: {mip_open}, cost {lakh(mip_cost)}")
print(f"Trying all 31 said:  {feasible.loc[0, 'open DCs']}, cost {lakh(feasible.loc[0, 'total'])}")

MIP says open: ['Delhi', 'Mumbai', 'Bengaluru', 'Kolkata'], cost Rs 33.8 lakh
Trying all 31 said:  Delhi + Mumbai + Bengaluru + Kolkata, cost Rs 33.8 lakh


Same answer. Here's which DC serves each city:

In [8]:
served = {c: max(SITES, key=lambda s: ship[s, c].varValue) for c in CITIES}
india_map(f"Best network: {', '.join(mip_open)} ({lakh(mip_cost)}/week)",
          lines=[(s, c) for c, s in served.items() if s != c])
pd.DataFrame({"pairs shipped": {s: round(sum(ship[s, c].varValue for c in CITIES)) for s in mip_open},
              "capacity": {s: capacity[s] for s in mip_open}})

,pairs shipped,capacity
Delhi,14500,30000
Mumbai,21200,30000
Bengaluru,19700,25000
Kolkata,9100,20000


Each DC serves its own region. Even the cities near Nagpur (Nagpur, Indore) are covered by Mumbai, the nearest *open* DC. Note that Mumbai, the most expensive site, is open: it sits next to 9,000 pairs of demand, plus Pune, Surat and Ahmedabad.

## 5. LP relaxation: what if we could open part of a DC?

How does the solver avoid trying every combination? Its first move is to **cheat on purpose**: let `open` be any number from 0 to 1 instead of only 0 or 1. That turns the MIP back into an ordinary LP, which solves very fast. This is called the **LP relaxation**.

In [9]:
_, model_lp, open_lp, ship_lp = solve_mip(relax=True)
lp_bound = pulp.value(model_lp.objective)

print(f"LP relaxation cost: {lakh(lp_bound)}   real best: {lakh(mip_cost)}")
pd.DataFrame({
    "open (fraction)": {s: round(open_lp[s].varValue, 3) for s in SITES},
    "pairs shipped": {s: round(sum(ship_lp[s, c].varValue for c in CITIES)) for s in SITES},
    "capacity": capacity,
})

LP relaxation cost: Rs 23.1 lakh   real best: Rs 33.8 lakh


,open (fraction),pairs shipped,capacity
Delhi,0.483,14500,30000
Mumbai,0.617,18500,30000
Bengaluru,0.588,14700,25000
Kolkata,0.400,8000,20000
Nagpur,0.251,8800,35000


**Look at the "open" column next to "pairs shipped".** Kolkata is 0.400 open and ships 8,000 of its 20,000 capacity: exactly 40%. The relaxation "rents" exactly the share of each DC it uses, so it pays only 40% of Kolkata's rent. That's impossible in real life, which is why the relaxation is so cheap (₹23.1 lakh).

**Why is a cheating answer useful?** It's a **lower bound**. The relaxation has *more* freedom than the real problem, so no real plan can ever cost less than ₹23.1 lakh. If we find a real plan costing ₹33.8 lakh, we know the best possible is somewhere between ₹23.1 and ₹33.8 lakh.

The distance between the best real plan found so far and the lower bound is the **MIP gap**:

> **gap = (best plan found − lower bound) ÷ best plan found**

Right now that's (33.8 − 23.1) ÷ 33.8 ≈ **32%**. The solver's job is to close it.

## 6. Branch and bound, by hand

**Branch and bound** closes the gap like this:

1. Solve the LP relaxation. If every `open` value is already 0 or 1, it's a real plan, so keep it if it's the best so far.
2. Otherwise, pick a DC that's open by a fraction (say Delhi = 0.48) and **branch**: solve two smaller problems, one with Delhi forced **open (1)** and one with Delhi forced **closed (0)**.
3. Repeat on each branch.
4. **Bound (prune):** if a branch's LP relaxation already costs more than the best real plan found so far, nothing inside that branch can beat it. Skip the whole branch.

Here it is on our problem. Each row is one LP relaxation solved:

In [10]:
def branch_and_bound(tight=False):
    """Plain depth-first branch and bound on the 'open' variables, using LP relaxations."""
    best = {"cost": math.inf, "open": None}
    log = []

    def visit(fixed, depth):
        stats, model, is_open, _ = solve_mip(relax=True, tight=tight, fixed=fixed)
        name = ", ".join(f"{s}={v}" for s, v in fixed.items()) or "(start: nothing fixed)"
        if stats.status != pulp.LpSolveStatus.Optimal:
            log.append({"depth": depth, "fixed so far": name, "LP bound": None, "what happened": "no feasible plan, stop"})
            return
        bound = pulp.value(model.objective)
        values = {s: is_open[s].varValue for s in SITES}
        if bound >= best["cost"] - 1e-6:
            log.append({"depth": depth, "fixed so far": name, "LP bound": bound,
                        "what happened": f"bound >= best so far ({lakh(best['cost'])}), prune"})
            return
        fractional = [s for s in SITES if 1e-6 < values[s] < 1 - 1e-6]
        if not fractional:
            best.update(cost=bound, open=[s for s in SITES if values[s] > 0.5])
            log.append({"depth": depth, "fixed so far": name, "LP bound": bound,
                        "what happened": "all whole numbers: new best plan"})
            return
        pick = max(fractional, key=lambda s: min(values[s], 1 - values[s]))
        log.append({"depth": depth, "fixed so far": name, "LP bound": bound,
                    "what happened": f"{pick} is {values[pick]:.2f} open, branch on it"})
        for v in (1, 0):
            visit({**fixed, pick: v}, depth + 1)

    visit({}, 0)
    return best, pd.DataFrame(log)


best, log = branch_and_bound()
log_view = log.assign(**{"LP bound": log["LP bound"].map(lambda v: "" if pd.isna(v) else lakh(v)),
                         "fixed so far": ["· " * d + f for d, f in zip(log["depth"], log["fixed so far"])]})
print(f"{len(log)} LP relaxations solved. Best plan: {best['open']}, {lakh(best['cost'])}")
log_view

31 LP relaxations solved. Best plan: ['Delhi', 'Mumbai', 'Bengaluru', 'Kolkata'], Rs 33.8 lakh


,depth,fixed so far,LP bound,what happened
0,0,(start: nothing fixed),Rs 23.1 lakh,"Delhi is 0.48 open, branch on it"
1,1,· Delhi=1,Rs 26.1 lakh,"Mumbai is 0.50 open, branch on it"
2,2,"· · Delhi=1, Mumbai=1",Rs 28.9 lakh,"Bengaluru is 0.59 open, branch on it"
3,3,"· · · Delhi=1, Mumbai=1, Bengaluru=1",Rs 31.0 lakh,"Kolkata is 0.40 open, branch on it"
4,4,"· · · · Delhi=1, Mumbai=1, Bengaluru=1, Kolkata=1",Rs 33.4 lakh,"Nagpur is 0.07 open, branch on it"
5,5,"· · · · · Delhi=1, Mumbai=1, Bengaluru=1, Kolkata=1, Nagpur=1",Rs 35.8 lakh,all whole numbers: new best plan
6,5,"· · · · · Delhi=1, Mumbai=1, Bengaluru=1, Kolkata=1, Nagpur=0",Rs 33.8 lakh,all whole numbers: new best plan
7,4,"· · · · Delhi=1, Mumbai=1, Bengaluru=1, Kolkata=0",Rs 33.7 lakh,"Nagpur is 0.23 open, branch on it"
8,5,"· · · · · Delhi=1, Mumbai=1, Bengaluru=1, Kolkata=0, Nagpur=1",Rs 35.6 lakh,"bound >= best so far (Rs 33.8 lakh), prune"
9,5,"· · · · · Delhi=1, Mumbai=1, Bengaluru=1, Kolkata=0, Nagpur=0",Rs 35.0 lakh,"bound >= best so far (Rs 33.8 lakh), prune"


**How to read the table:** the number of `·` marks and the `depth` column show how deep in the search tree each LP is. Follow it from the top:
- At the start, Delhi is 0.48 open, so the search branches on Delhi. It tries Delhi = 1 first and keeps going down.
- Five levels down, everything is a whole number, so the first real plans appear: all 5 DCs (₹35.8 lakh), then 4 DCs without Nagpur (**₹33.8 lakh**, the eventual answer).
- From then on, any branch whose LP bound is already above ₹33.8 lakh is **pruned**, and nothing below it is explored.

**But notice:** it took **31 LP relaxations**, almost as many as trying all 31 combinations. The pruning barely helped. Why? The bounds were weak: the relaxation started at ₹23.1 lakh, far below the real answer, so many branches *looked* promising and had to be checked.

**This is why MIPs get slow:** every yes/no variable can double the tree, and weak bounds mean less pruning. With 80 yes/no choices, the tree could have 2⁸⁰ leaves.

## 7. Write the model better, search far less

Add one rule that seems to say nothing new:

> **`ship[s, c]` ≤ demand of c × `open[s]`**: a DC can't send a city more than that city needs, and nothing at all if closed.

Every real plan already obeys this. But in the relaxation it stops the cheating: a DC that's 0.4 open can now send each city at most **40% of that city's demand**. To serve a city fully, the relaxation has to "open" DCs much more.

In [11]:
_, model_tight_lp, open_tight, _ = solve_mip(relax=True, tight=True)
best_t, log_t = branch_and_bound(tight=True)

print("LP relaxation 'open' values:", {s: round(open_tight[s].varValue, 3) for s in SITES})
print(f"LP relaxation cost: {lakh(pulp.value(model_tight_lp.objective))}   real best: {lakh(mip_cost)}")
print(f"Branch and bound: {len(log_t)} LP relaxation(s) instead of {len(log)}")

LP relaxation 'open' values: {'Delhi': 1.0, 'Mumbai': 1.0, 'Bengaluru': 1.0, 'Kolkata': 1.0, 'Nagpur': 0.0}
LP relaxation cost: Rs 33.8 lakh   real best: Rs 33.8 lakh
Branch and bound: 1 LP relaxation(s) instead of 31


With the tighter rule, the LP relaxation gives **exactly** the real answer, with every `open` already 0 or 1. Branch and bound finishes after **1 LP instead of 31**.

**The lesson:** two models with the *same* correct answers can be very different for the solver. The one whose LP relaxation is closer to the real answer (a **tighter** or **stronger formulation**) prunes far more.

The tradeoff: the tighter model has 5 × 20 = 100 extra rules. On big problems, modern solvers like SCIP and HiGHS add many such rules themselves (called **cuts**), so the gain is smaller there. It's still the first thing to try when a MIP is slow.

## 8. The same model in OR-Tools

The job description names **OR-Tools**, Google's optimization library. The model reads almost line for line like the PuLP one: `BoolVar` is a binary variable, `NumVar` a continuous one, and `solver.Add` adds a rule. We use its **SCIP** solver, a strong open-source MIP solver.

The function below can also build random problems of any size, which we'll use in step 9.

In [12]:
def solve_ortools(n_sites=None, n_cities=None, seed=0, time_limit_s=None):
    """Facility location in OR-Tools with SCIP. With n_sites/n_cities, builds a random instance."""
    if n_sites is None:
        sites, places, dem, cap, fix = SITES, CITIES, demand, capacity, fixed_cost
        cost = {(s, c): ship_cost.loc[s, c] for s in SITES for c in CITIES}
    else:
        rng = np.random.default_rng(seed)
        site_xy, city_xy = rng.uniform(0, 100, (n_sites, 2)), rng.uniform(0, 100, (n_cities, 2))
        sites, places = range(n_sites), range(n_cities)
        dem = dict(enumerate(rng.integers(500, 3000, n_cities).tolist()))
        avg = sum(dem.values()) / n_sites
        cap = dict(enumerate(rng.integers(int(2 * avg), int(4 * avg), n_sites).tolist()))
        fix = dict(enumerate(rng.integers(200_000, 600_000, n_sites).tolist()))
        dist = np.linalg.norm(site_xy[:, None, :] - city_xy[None, :, :], axis=2)
        cost = {(s, c): 5 + 0.5 * float(dist[s, c]) for s in sites for c in places}

    solver = pywraplp.Solver.CreateSolver("SCIP")
    if time_limit_s:
        solver.SetTimeLimit(int(time_limit_s * 1000))

    is_open = {s: solver.BoolVar(f"open_{s}") for s in sites}
    ship = {(s, c): solver.NumVar(0, solver.infinity(), f"ship_{s}_{c}") for s in sites for c in places}

    solver.Minimize(sum(fix[s] * is_open[s] for s in sites) + sum(cost[k] * ship[k] for k in ship))
    for c in places:
        solver.Add(sum(ship[s, c] for s in sites) >= dem[c])
    for s in sites:
        solver.Add(sum(ship[s, c] for c in places) <= cap[s] * is_open[s])

    start = time.time()
    status = solver.Solve()
    seconds = time.time() - start
    best, bound = solver.Objective().Value(), solver.Objective().BestBound()
    return {
        "status": "optimal" if status == pywraplp.Solver.OPTIMAL else "stopped at time limit",
        "cost": best, "bound": bound, "gap %": 100 * (best - bound) / best, "seconds": seconds,
        "open": [s for s in sites if is_open[s].solution_value() > 0.5],
    }


ort = solve_ortools()
print(f"OR-Tools / SCIP: open {ort['open']}, cost {lakh(ort['cost'])}, solved in {ort['seconds']:.2f} s")

OR-Tools / SCIP: open ['Delhi', 'Mumbai', 'Bengaluru', 'Kolkata'], cost Rs 33.8 lakh, solved in 0.03 s


## 9. How does solve time grow?

Let's make random problems from 10 sites × 50 cities up to 80 sites × 400 cities, and give the solver at most **20 seconds** each. If it runs out of time, it stops and reports the best plan found so far and the gap.

In [13]:
sizes = [(10, 50), (20, 100), (30, 150), (40, 200), (60, 300), (80, 400)]
runs = []
for n_sites, n_cities in sizes:
    r = solve_ortools(n_sites, n_cities, time_limit_s=20)
    runs.append({"size": f"{n_sites} sites x {n_cities} cities", "yes/no choices": n_sites,
                 "seconds": round(r["seconds"], 2), "gap %": round(r["gap %"], 2), "status": r["status"]})
runs = pd.DataFrame(runs)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Solve time", "MIP gap when the solver stopped"))
fig.add_trace(go.Bar(x=runs["size"], y=runs["seconds"], hovertemplate="%{x}<br>%{y:.1f} s<extra></extra>"), row=1, col=1)
fig.add_trace(go.Bar(x=runs["size"], y=runs["gap %"], hovertemplate="%{x}<br>gap %{y:.2f}%<extra></extra>"), row=1, col=2)
fig.add_hline(y=20, line_dash="dot", annotation_text="time limit", row=1, col=1)
fig.update_yaxes(title_text="seconds", row=1, col=1)
fig.update_yaxes(title_text="gap (%)", row=1, col=2)
fig.update_layout(height=430, width=1000, showlegend=False, template="plotly_white")
fig.show()
runs

,size,yes/no choices,seconds,gap %,status
0,10 sites x 50 cities,10,0.06,0.00,optimal
1,20 sites x 100 cities,20,0.67,0.00,optimal
2,30 sites x 150 cities,30,1.89,0.00,optimal
3,40 sites x 200 cities,40,6.13,0.00,optimal
4,60 sites x 300 cities,60,13.69,0.00,optimal
5,80 sites x 400 cities,80,20.04,1.45,stopped at time limit


**Read it:** solve time climbs much faster than the problem size. The small problems finish in well under a second and the mid-sized ones in seconds. The biggest one hits the 20-second limit before the solver can **prove** its plan is the best, so it reports a gap (the right chart).

**What does that gap mean?** If the gap is about 1.5%, the plan in hand is **at most 1.5% more expensive than the best plan possible**. It might already *be* the best; the solver just hasn't proved it yet.

**When would you stop early?** Nearly always, in practice. Planners need an answer by a deadline. A plan proven to be within 1% of optimal, delivered by 9 am, beats a perfect plan delivered at noon. Especially since the demand forecast feeding the model is itself off by much more than 1%. Typical settings are a time limit plus a gap target ("stop at 1%").

## Recap: say these out loud

1. **LP vs MIP?** A MIP has some whole-number or yes/no variables. Real choices like "open this DC" or "run this line" need them.
2. **Why are MIPs hard?** Every yes/no choice can double the number of possibilities: 5 sites = 32, 80 sites = 2⁸⁰. There's no known fast method that works for all MIPs (they're *NP-hard*).
3. **How do you model a fixed cost?** A binary `open`, the fixed cost × `open` in the objective, and a linking rule: quantity ≤ capacity × `open` (big-M with M = capacity).
4. **What's the LP relaxation for?** Let binaries be fractions. It's fast and gives a lower bound: no real plan can be cheaper.
5. **How does branch and bound work?** Pick a fractional variable, try 0 and 1 as two branches, and prune any branch whose bound is already worse than the best plan found.
6. **What does a 1% gap mean?** The plan is at most 1% worse than the best possible. You'd usually stop there.
7. **What makes a MIP faster?** A tighter formulation (relaxation closer to the truth), good time limits and gap targets, and a good starting plan.
8. **The network answer here:** 4 metro DCs, ₹33.8 lakh a week. The central, cheap site loses because it's medium-far from everyone.

**Words from today**

| Word | Meaning |
|---|---|
| Binary variable | Can only be 0 or 1 (no / yes) |
| MIP | LP plus some whole-number variables |
| Fixed cost | Paid in full if you do something at all (open, run, ship), however much you do |
| Linking / big-M constraint | quantity ≤ M × binary: forces the quantity to 0 when the binary is 0 |
| LP relaxation | The MIP with binaries allowed to be fractions. Gives a lower bound (for minimizing) |
| Branch and bound | Split on a fractional variable, solve both sides, prune sides that can't win |
| MIP gap | (best plan found − bound) ÷ best plan found |
| Tight / strong formulation | A model whose LP relaxation is close to the real answer, so it prunes more |
| Cuts | Extra rules the solver adds itself to tighten the relaxation |

**Try it yourself:**
1. Cut Nagpur's fixed cost to ₹1 lakh. *Predict:* does Nagpur get opened? Which DC closes?
2. Double the per-km shipping cost (0.05 → 0.10). *Predict:* more DCs or fewer?